# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiknaxTheGreek/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Primary task: ranking / scoring in a hybrid ML pipeline.**

The system must answer: **which content pages should get attention first, why, and what action should be considered?** That makes ranking/scoring the final task.

The ranking is built in stages:

1. **Signal analysis** creates a leakage-safe feature set from observed search, traffic, engagement, freshness, and content data.
2. **Clustering** groups similar pages into performance archetypes and gives each page a fair peer group.
3. **Peer-relative analysis** measures how unusual a page is compared with similar pages, for example whether its CTR or engagement is weak for its archetype.
4. **Classification / prediction** estimates a future observed performance state using only information available before the decision point.
5. **Impact estimation** combines predicted risk, expected size of the change, and page exposure/value.
6. **Ranking / scoring** orders pages by expected adverse impact.
7. **Action generation** uses the archetype, predicted direction, and strongest abnormal signals to produce reason codes and a suggested action.

In short:

**observed signals → archetype → relative abnormalities → future prediction → expected impact → ranked priority → suggested action**

The suggested action is an evidence-based **intervention hypothesis**, not a proven causal effect. We can test prediction quality retrospectively and later test whether the recommended action causes improvement with a controlled or otherwise valid causal study.

In [1]:
import pandas as pd
from pathlib import Path

candidate_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]

data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Unique content items: {df['content_id'].nunique():,}")
print(f"Pseudonymized clients: {df['client_id'].nunique():,}")

assert df['content_id'].nunique() == len(df), (
    "Expected one row per pseudonymized content item."
)

methodology = [
    "signal analysis",
    "clustering / peer context",
    "archetype-relative deviations",
    "future-state prediction",
    "impact estimation",
    "ranking / scoring",
    "action hypothesis",
]
print("Methodology:", " -> ".join(methodology))


Rows: 30,000
Columns: 44
Unique content items: 30,000
Pseudonymized clients: 32
Methodology: signal analysis -> clustering / peer context -> archetype-relative deviations -> future-state prediction -> impact estimation -> ranking / scoring -> action hypothesis


## 2. Target or proxy

Different stages use different learning objects.

### Clustering

Clustering has **no target column**. It creates performance archetypes and peer baselines from the training data.

### Supervised prediction

The main supervised target should be an **observed future outcome**, not a manually created priority score or action label.

The intended structure is:

**past feature window → decision point → non-overlapping future outcome window**

The later warehouse target will be a future decline outcome such as `future_decline_30d`, built from observed search performance after the decision point. A continuous future-change value will also keep the size of the movement, so a small decline and a severe decline are not treated as equal.

The exact decline threshold, persistence rule, and minimum-volume floor are **not fixed in this notebook**. They will be defined in the data-contract stage and tested for sensitivity before model training.

### Starter-data proxy

The 30,000-row starter CSV is only a trailing-90-day snapshot, so it cannot provide a true future target. For Assignment 3, the transparent proxy is:

[
\text{decline\_proxy}=1
\quad \text{if} \quad
\texttt{trend\_direction = "down"}
]

This is a **defined current-window proxy**, not a future ground-truth outcome and not proof that a page needs intervention.

Because `trend_direction` comes from `trend_pct`, neither field can be used as a predictive feature when this proxy is the target.

### Ranking and action

The final ranking will not learn a hand-written priority label. Conceptually:

[
\text{priority}
\propto
P(\text{future decline})
\times
E(\text{decline magnitude})
\times
\text{measured exposure/value}
]

The exact scaling will be fixed later and validated rather than assigned arbitrary weights.

The action stage maps the page's archetype, predicted future state, and strongest peer-relative abnormalities to a **recommended intervention hypothesis** that can later be tested.

In [2]:
# Transparent starter-data proxy for Assignment 3 framing.
required_proxy_columns = {
    "content_id",
    "trend_direction",
    "trend_pct",
    "impressions_prev_30d",
    "impressions_last_30d",
}
missing = sorted(required_proxy_columns.difference(df.columns))
assert not missing, f"Missing required proxy columns: {missing}"

proxy_frame = df[[
    "content_id",
    "impressions_prev_30d",
    "impressions_last_30d",
    "trend_pct",
    "trend_direction",
]].copy()

proxy_frame["decline_proxy"] = (
    proxy_frame["trend_direction"].str.lower().eq("down").astype("int8")
)

print("Starter proxy: decline_proxy = 1 when trend_direction == 'down'.")
print("This is a current-window proxy, not a future causal or intervention label.\n")
print(proxy_frame["decline_proxy"].value_counts().sort_index())
print(f"Proxy positive rate: {proxy_frame['decline_proxy'].mean():.1%}\n")

display(proxy_frame.head(10))

print("Excluded from predictive features for this proxy: ['trend_direction', 'trend_pct']")

# Later warehouse target structure (not fabricated from this snapshot):
target_schema = pd.DataFrame({
    "field": [
        "future_decline_30d",
        "future_change_magnitude",
    ],
    "role": [
        "future-state classification target",
        "future-movement magnitude outcome",
    ],
    "available_in_starter_snapshot": [False, False],
})
display(target_schema)


Starter proxy: decline_proxy = 1 when trend_direction == 'down'.
This is a current-window proxy, not a future causal or intervention label.

decline_proxy
0    13738
1    16262
Name: count, dtype: int64
Proxy positive rate: 54.2%



,content_id,impressions_prev_30d,impressions_last_30d,trend_pct,trend_direction,decline_proxy
0,content_304f48230142,987,578,-41.4,down,1
1,content_a1fb4e703a9e,5915,2501,-57.7,down,1
2,content_9aa793d4d895,6089,2382,-60.9,down,1
3,content_331d6c4de07b,4206,3626,-13.8,stable,0
4,content_d99b7a2d90ca,6452,4211,-34.7,down,1
5,content_d4084a4bc775,1009,617,-38.9,down,1
6,content_9a34b442b552,13,1,-92.3,down,1
7,content_a63219c6e95a,632,636,0.6,stable,0
8,content_5e6c160719bc,13828,5696,-58.8,down,1
9,content_c27558df2b0c,356,252,-29.2,down,1


Excluded from predictive features for this proxy: ['trend_direction', 'trend_pct']


,field,role,available_in_starter_snapshot
0,future_decline_30d,future-state classification target,false
1,future_change_magnitude,future-movement magnitude outcome,false


## 3. Success metric

**Primary end-to-end metric: Precision@K for the ranked human-review queue.**

The final decision is not “is this page positive or negative?” It is **which pages should limited human review capacity inspect first?** Therefore the headline metric must evaluate the top of the ranked queue.

For any final evaluation, all metrics must be computed on an **honest future outcome window**, using only features available before the decision point. Model and baseline must use the **same rows, same target, same split and same K**. The rule baseline is frozen before model comparison. The starter `decline_proxy` is only for Assignment-3 plumbing; it is not evidence of future predictive skill.

### Metric contract: every task and sub-task

| Stage / sub-task | Exact metric | Baseline / comparator | Pass rule | What failure tells us |
|---|---|---|---|---|
| **Signal analysis — data support** | Missing-rate table and valid `n` for every tested slice | Pre-declared sample floor: about 50 rows per main bucket and 30 for cross-cuts | No substantive claim is made from a slice below the floor; missingness is reported by relevant content group | A “signal” may only be sparse data, instrumentation, or category-driven missingness |
| **Signal analysis — binary-outcome usefulness** | **Average Precision (AP) lift**: `AP(signal) - future-outcome prevalence`, evaluated out of sample; signal direction chosen on training data only | No-skill AP = held-out positive prevalence | AP lift > 0 and the grouped/client-aware bootstrap 95% CI for the lift does not cross 0 | The signal does not reliably rank future adverse outcomes above chance |
| **Signal analysis — continuous-outcome usefulness** | **Spearman rho** between the signal and future-change magnitude, with grouped bootstrap CI | rho = 0 | CI excludes 0 and the direction is consistent with the training-period hypothesis | Apparent association is unstable, driven by outliers, or does not reproduce |
| **Clustering — geometric separation** | **Silhouette score** on standardized leakage-safe clustering features | 0 = no average separation advantage | Silhouette > 0 on validation data | Pages are not meaningfully closer to their assigned archetype than to alternatives |
| **Clustering — stability** | **Adjusted Rand Index (ARI)** across bootstrap/refit clusterings on common pages | Random/relabelled clustering has expected ARI near 0 | Bootstrap ARI is consistently above the random/relabelled reference | Archetypes are unstable and may be artifacts of one sample or initialization |
| **Clustering — decision usefulness** | Difference in **peer-abnormality Lift@10%** using cluster peers versus a global-peer version | Same abnormality method without clusters | Cluster-based Lift@10% > global-peer Lift@10% on the same validation rows | Clustering adds complexity but no useful peer context; remove it |
| **Peer-relative abnormality — concentration** | **Lift@10%** = future adverse-outcome rate in the most-abnormal 10% / overall future-outcome rate | 1.0 = no enrichment over prevalence | Lift@10% > 1 and > the corresponding raw/global-deviation baseline; grouped-bootstrap CI for the improvement excludes 0 | “Abnormal” pages are not actually enriched for later problems |
| **Peer-relative abnormality — ordering** | **Spearman rho** between pre-declared abnormality decile (1–10) and future adverse-outcome rate by decile | rho = 0 | rho > 0 with no single tiny bucket driving the result | Top-decile lift may be a one-bin accident rather than a graded relationship |
| **Future-state classification — discrimination** | **Average Precision / PR-AUC** on held-out future labels | No-skill AP = future-outcome prevalence | AP > prevalence and grouped-bootstrap 95% CI for `AP - prevalence` excludes 0 | Classifier cannot reliably rank true future events above chance |
| **Future-state classification — calibration** | **Brier Skill Score** = `1 - Brier(model) / Brier(prevalence predictor)` | Constant probability equal to training prevalence | Brier Skill Score > 0 after calibration if calibration is needed | Raw probabilities are not trustworthy enough to multiply into impact; recalibrate or do not use them as probabilities |
| **Future-state classification — operating point** | Precision and recall at the **pre-declared review threshold / capacity** | Same capacity for baseline and model | Report both; do not move the threshold after seeing test results | A good global PR-AUC may still produce an unusable operating point |
| **Future-change magnitude — absolute error** | **MAE ratio** = `MAE(model) / MAE(training-median baseline)` | Training-set median prediction, which is the proper constant baseline for MAE | MAE ratio < 1 on held-out future magnitude | Magnitude model does not improve on a trivial constant forecast |
| **Future-change magnitude — ordering** | **Spearman rho** between predicted and realized future adverse magnitude | rho = 0 | rho > 0 with grouped-bootstrap CI excluding 0 | Even if MAE improves slightly, the model does not correctly order severity |
| **Impact estimation — top-weighted severity ranking** | **NDCG@K** using frozen realized adverse-impact magnitude as graded relevance | Compare with (a) risk-only ranking and (b) transparent rule baseline | Impact NDCG@K must beat both comparators on the same rows | Multiplying risk, magnitude and exposure is not adding useful severity information |
| **Impact estimation — global ordering** | **Spearman rho** between estimated and realized adverse impact | rho = 0 | rho > 0 with grouped-bootstrap CI excluding 0 | Impact scores may look good at K only because of a few pages |
| **Final ranking — primary** | **Precision@50** = true future adverse outcomes among top 50 / 50 | Frozen transparent rule baseline at K=50; also print future-outcome prevalence | `P@50_model - P@50_baseline > 0`, with grouped-bootstrap CI for the difference excluding 0 | Learned ranking has not earned its extra complexity |
| **Final ranking — anti-cherry-pick depths** | **Precision@20 and Precision@100** | Same rule baseline at each K | Always report alongside P@50; no requirement that every K wins, but losses must be disclosed and explained | A claimed win may exist only at a hand-picked queue depth |
| **Final ranking — coverage** | **Recall@K** | Same K | Report with Precision@K | High precision may be obtained by catching only a tiny fraction of all relevant pages |
| **Final ranking — severity-aware diagnostic** | **NDCG@K** using realized adverse-impact magnitude | Same K and comparator rankings | Report with Precision@K | Binary relevance can hide whether the queue captures the most severe cases |
| **Reason codes — traceability gate** | **Traceability rate** = recommendations whose reason codes can be reproduced from stored pre-decision signals / all recommendations | Required system property | **100%** | Some recommendations cannot be audited back to evidence and must not be shipped |
| **Action generation — contradiction gate** | **Unsupported-action rate** = actions whose required evidence is absent or contradicted / all suggested actions | Required system property | **0%** | The action mapper is generating advice not supported by its own measured evidence |
| **Action generation — human validity (only when labels exist)** | **Reviewer appropriateness rate** plus **Cohen’s kappa** for inter-rater agreement when two reviewers are available | Predefined review rubric; not available in the starter data | Report only when real reviewer judgements exist; otherwise mark **not yet measurable** | We cannot honestly claim action “accuracy” without a human or intervention ground truth |
| **Action efficacy — causal follow-up, outside this predictive assignment** | Controlled change in the chosen outcome versus control / valid causal comparator | No-intervention or appropriate control group | Must be established in a later controlled or otherwise defensible causal study | Predictive success does not prove that the recommended intervention causes improvement |

### Headline success rule

For the final ranked queue:

$$
\Delta P@50 =
P@50_{\text{model}} -
P@50_{\text{rule baseline}}
$$

The learned ranking earns its place only if **Delta Precision@50 is positive on the honest held-out future data and its grouped/client-aware uncertainty interval excludes zero**. We also report the future-outcome base rate, Lift@50, Precision@20, Precision@100, Recall@K and NDCG@K so that a single flattering number cannot hide a weak queue.

`K = 50` is the **pre-declared Assignment-3 reporting depth**, not a claim that 50 is FlyRank's true business capacity. If a real review budget is supplied later, that operational K replaces it prospectively; we do not choose K after seeing test performance.

### Validation rules that apply to every metric

1. **Past → decision point → future:** no overlapping feature/outcome windows.
2. **Grouped/time-aware validation:** repeated client/site structure must not leak across validation; final performance must mimic deployment through time.
3. **Base rate beside every classification/ranking metric:** AP or Precision@K without prevalence can be misleading.
4. **Grouped uncertainty:** resample at the client/group level rather than pretending correlated pages are independent.
5. **Same comparison frame:** model and baseline get identical rows, labels, K and evaluation dates.
6. **No test-set tuning:** feature direction, cluster count, thresholds, calibration and K are fixed using training/validation data before the final holdout.
7. **No stage survives by default:** if a stage fails its relevant gate or adds no downstream value, simplify or remove it.

This makes the pipeline falsifiable: every stage has a number that can expose failure, and the final model is only justified if the full ranked decision system improves over the simpler baseline.

In [ ]:
# Explicit metric contract: primary metrics, comparators, and hard gates.
import numpy as np
import pandas as pd

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    if len(y_true) != len(scores):
        raise ValueError("y_true and scores must have the same length")
    if not 1 <= k <= len(y_true):
        raise ValueError("k must be between 1 and the number of rows")
    order = np.argsort(-scores, kind="stable")
    return float(y_true[order[:k]].mean())

def lift_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    base_rate = float(y_true.mean())
    if base_rate == 0:
        return np.nan
    return precision_at_k(y_true, scores, k) / base_rate

metric_contract = pd.DataFrame([
    ("signal", "binary usefulness", "AP lift over prevalence", "AP - prevalence > 0; grouped-bootstrap CI excludes 0"),
    ("signal", "continuous usefulness", "Spearman rho", "rho CI excludes 0 and direction reproduces"),
    ("clustering", "separation", "silhouette score", "silhouette > 0 on validation data"),
    ("clustering", "stability", "bootstrap Adjusted Rand Index", "above random/relabelled reference"),
    ("clustering", "downstream usefulness", "peer Lift@10% delta", "cluster-peer lift > global-peer lift"),
    ("peer abnormality", "concentration", "Lift@10%", "> 1 and > raw/global-deviation baseline"),
    ("peer abnormality", "ordering", "Spearman rho across decile event rates", "rho > 0"),
    ("classification", "discrimination", "Average Precision / PR-AUC", "AP > prevalence; delta CI excludes 0"),
    ("classification", "calibration", "Brier Skill Score", "> 0 versus prevalence predictor"),
    ("classification", "operating point", "precision + recall at fixed capacity", "report at pre-declared threshold"),
    ("magnitude", "absolute error", "MAE ratio", "MAE(model) / MAE(training-median baseline) < 1"),
    ("magnitude", "ordering", "Spearman rho", "rho > 0; grouped-bootstrap CI excludes 0"),
    ("impact", "top-weighted severity", "NDCG@K", "beats risk-only and rule-baseline rankings"),
    ("impact", "global ordering", "Spearman rho", "rho > 0; grouped-bootstrap CI excludes 0"),
    ("final ranking", "primary", "Precision@50", "beats frozen rule baseline; delta CI excludes 0"),
    ("final ranking", "anti-cherry-pick", "Precision@20 + Precision@100", "always report alongside P@50"),
    ("final ranking", "coverage", "Recall@K", "report beside Precision@K"),
    ("final ranking", "severity diagnostic", "NDCG@K", "report using realized adverse impact"),
    ("reason codes", "traceability", "traceability rate", "must equal 100%"),
    ("action", "contradiction", "unsupported-action rate", "must equal 0%"),
    ("action", "human validity", "appropriateness rate + Cohen's kappa", "not measurable until reviewer labels exist"),
], columns=["stage", "sub_task", "metric", "success_gate"])

reporting_depths = (20, 50, 100)
primary_k = 50
starter_proxy_rate = float(proxy_frame["decline_proxy"].mean())

display(metric_contract)
print(f"Pre-declared primary reporting depth: K = {primary_k}")
print(f"Required anti-cherry-pick depths: {reporting_depths}")
print(f"Starter proxy prevalence (plumbing only): {starter_proxy_rate:.1%}")
print("Primary final gate: Precision@50(model) > Precision@50(frozen rule baseline)")
print("and the grouped/client-aware bootstrap CI for the difference must exclude 0.")
print("Every stage is removable if it fails its own gate or adds no downstream value.")


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.